# 09. Building a batch-processing pipeline

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner  
**Estimated study time:** 30–60 minutes

## What you will learn
- Wrap analysis steps in a reusable function
- Process multiple images consistently
- Prepare a safe folder-based batch workflow

> **Course habit:** run one cell at a time, inspect the result, and change one parameter before moving on.

## Turn one working analysis into a function

A pipeline is easier to reuse and test when repeated operations are wrapped in a short function.

In [ ]:
import numpy as np
import pandas as pd
import skimage as ski

def analyze_objects(image, min_size=100, sigma=1.0):
    """Segment bright objects and return labels plus a measurement table."""
    smooth = ski.filters.gaussian(image, sigma=sigma)
    threshold = ski.filters.threshold_otsu(smooth)
    mask = smooth > threshold
    mask = ski.morphology.remove_small_objects(mask, max_size=min_size - 1)
    labels = ski.measure.label(mask)
    props = ski.measure.regionprops_table(labels, intensity_image=image, properties=('label','area','mean_intensity','eccentricity'))
    return labels, pd.DataFrame(props)

## Run the same pipeline on several images

In [ ]:
images={'coins':ski.data.coins(),'camera_crop':ski.data.camera()[100:400,100:400],'cell':ski.data.cell()}
summaries=[]
for name,image in images.items():
    labels,table=analyze_objects(image)
    summaries.append({'image':name,'n_objects':len(table),'mean_area':table['area'].mean() if len(table) else np.nan})
summary_df=pd.DataFrame(summaries)
print(summary_df)

## Batch files from a folder

In [ ]:
from pathlib import Path
import imageio.v3 as iio

data_dir=Path('../data/raw')
image_files=sorted(data_dir.glob('*.tif')) if data_dir.exists() else []
if not image_files:
    print('No TIFF files found yet. Add your own files to ../data/raw when ready.')
for path in image_files:
    image=iio.imread(path)
    labels,table=analyze_objects(image)
    print(path.name,'->',len(table),'objects')

## Practical checks before batch processing
- Test representative images first.
- Save parameters with the results.
- Keep original files unchanged.
- Inspect masks from good and difficult cases.
- Do not assume one setting works for every acquisition condition.

## Exercise
Add `solidity` to the function and output table.